# Interpreting sequence features of *in vivo* enzymes

This notebook runs the InterPLM pipeline to perform an interpretability analysis of *in vivo*-like homomeric enzymes from *E. coli* (iML1515 metabolic model). 

The analysis aims to identify sequence features that distinguish enzymes based on their $\eta$ variability (coefficient of variation in enzyme saturation levels across different metabolic conditions). Specifically, we:

1. **Stratify enzymes** into high and low variance groups based on their $\eta$ CV values (using 75th and 25th percentiles)
2. **Extract ESM-2 embeddings** from enzyme sequences using the esm2_t6_8M_UR50D model (layer 4)
3. **Compute SAE (Sparse Autoencoder) activations** to identify interpretable features (10,420 features total)
4. **Perform statistical analysis** comparing feature activations between high and low variance groups

The goal is to discover which molecular features, as learned by the protein language model, correlate with metabolic variability and potentially reveal functional constraints or regulatory mechanisms.

## 1. Imports

In [ ]:
# automatic module reloading
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import subprocess
from datetime import datetime
import torch
import pickle
from scipy import stats

# Add paths & directories for imports
sys.path.append(str(Path("..").resolve()))
sys.path.append(str((Path("..") / "interplm").resolve()))

# Data directory
data_dir = Path("..") / "data" / "in_vivo_variability"

# Create directories
embeddings_dir = data_dir / "embeddings"
logs_dir = data_dir / "logs"
sae_activations_path = data_dir / "sae_activations"

embeddings_dir.mkdir(parents=True, exist_ok=True)
logs_dir.mkdir(parents=True, exist_ok=True)
sae_activations_path.mkdir(parents=True, exist_ok=True)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Prepare in vivo enzyme dataframe

In [ ]:
# Load in vivo dataset
df_path = data_dir / "raw" / "iml1515_homomeric_kmax_pFBA_variability.csv"
df = pd.read_csv(df_path)

# Stratify into high vs low variance groups
high_var_threshold = df['eta_cv'].quantile(0.75)
low_var_threshold = df['eta_cv'].quantile(0.25)

# Groups: medium, high, low
df['variance_group'] = 'medium'
df.loc[df['eta_cv'] >= high_var_threshold, 'variance_group'] = 'high'
df.loc[df['eta_cv'] <= low_var_threshold, 'variance_group'] = 'low'

# Deduplicate by gene (keeping first)
print(f"Total rows before deduplication: {len(df)}")
df_unique = df.drop_duplicates(subset='gene', keep='first')
print(f"Total rows after deduplication: {len(df_unique)}")
print(f"{len(df) - len(df_unique)} duplicate genes removed")

# Show breakdown by group
print(f"\nHigh variance: {(df_unique['variance_group']=='high').sum()} enzymes")
print(f"Low variance: {(df_unique['variance_group']=='low').sum()} enzymes")
print(f"Medium variance: {(df_unique['variance_group']=='medium').sum()} enzymes")

# Save deduplicated version
df_unique.to_csv((data_dir / "processed" / "iml1515_variability_grouped.csv"), index=False)

# Update df to use the deduplicated version
df = df_unique

Total rows before deduplication: 377
Total rows after deduplication: 199
178 duplicate genes removed

High variance: 61 enzymes
Low variance: 54 enzymes
Medium variance: 84 enzymes


In [29]:
# Create FASTA files for embedding
for group in ['high', 'low']:
    group_df = df[df['variance_group'] == group]
    with open((data_dir / "processed" / f"iml1515_{group}_variance.fasta"), 'w') as f:
        for _, row in group_df.iterrows():
            f.write(f">{row['gene']}\n{row['sequence']}\n")

print(f"High variance: {(df['variance_group']=='high').sum()} enzymes")
print(f"Low variance: {(df['variance_group']=='low').sum()} enzymes")

High variance: 61 enzymes
Low variance: 54 enzymes


## 3. Extract ESM-2 Embeddings

In [27]:
from interplm.esm.fasta_to_sae_dataset import embed_fasta_file_for_all_layers

groups = ["high", "low"]
esm_model = "esm2_t6_8M_UR50D"
layer = 4

for group in groups:
    fasta_path = data_dir / "processed" / f"iml1515_{group}_variance.fasta"
    out_dir = embeddings_dir / group
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[{group}] Processing {fasta_path}...")
    embed_fasta_file_for_all_layers(
        esm_model_name=esm_model,
        fasta_file=fasta_path,
        output_dir=out_dir,
        layers=[layer],
        shard_num=0,
        corrupt_esm=False
    )
    print(f"[{group}] Done.")


[high] Processing ../data/in_vivo_variability/processed/iml1515_high_variance.fasta...
Read ../data/in_vivo_variability/processed/iml1515_high_variance.fasta with 61 sequences


Processing batches: 100%|██████████| 28/28 [00:01<00:00, 24.00it/s]


Saved activations for layer 4, shard 0 to ../data/in_vivo_variability/embeddings/high/layer_4/shard_0/activations.pt
[high] Done.

[low] Processing ../data/in_vivo_variability/processed/iml1515_low_variance.fasta...
Read ../data/in_vivo_variability/processed/iml1515_low_variance.fasta with 54 sequences


Processing batches: 100%|██████████| 21/21 [00:00<00:00, 60.30it/s]


Saved activations for layer 4, shard 0 to ../data/in_vivo_variability/embeddings/low/layer_4/shard_0/activations.pt
[low] Done.


# 4. Load pretrained models (ESM and SAEs)

In [36]:
from interplm.sae.inference import load_sae_from_hf
from transformers import AutoTokenizer, EsmModel

# Load SAE (using layer 4 from ESM-2-8M)
sae = load_sae_from_hf(plm_model="esm2-8m", plm_layer=4)

# Load ESM-2 model
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
esm_model = EsmModel.from_pretrained("facebook/esm2_t6_8M_UR50D")
esm_model = esm_model.to('cuda')


/nfs/homes/chinasse/miniconda3/envs/interplm/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
Some weights of EsmModel were not initialized from the model checkpoint at facebook/esm2_t6_8M_UR50D and are newly initialized: ['esm.pooler.dense.bias', 'esm.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# 5. Get SAE activations

The `get_sae_activations` function returns a dictionary with 4 keys:

```python
{
    'protein_id': <string>,
    'max_activations': <numpy array>, shape (10420,)
    'mean_activations': <numpy array>, shape (10420,)
    'n_active_features': <integer>
}
```

The contents are PER enzyme sequence (indexed by `protein_id`):
- `max_activations`: For each of the 10,420 SAE features, this is the maximum activation across all positions in the protein sequence
- `mean_activations`: For each of the 10,420 SAE features, this is the mean activation across all positions in the protein sequence
- `n_active_features`: How many of the 10,420 SAE features are present (above threshold of 0.15)

In [44]:
def get_sae_activations(sequence, protein_id, threshold=0.15):
    """Extract SAE feature activations for a protein sequence"""
    # Tokenize
    inputs = tokenizer(sequence, return_tensors="pt", padding=True)
    inputs = {k: v.to('cuda') for k, v in inputs.items()}  # Move tokens to GPU
    
    # Get ESM embeddings
    with torch.no_grad():
        outputs = esm_model(**inputs, output_hidden_states=True)
        # Layer 4 embeddings (index 4 in hidden_states)
        layer_4_embeds = outputs.hidden_states[4][0]  # [seq_len, 320]
        #print('ESM embeddings shape:')
        #print(layer_4_embeds.shape)
    
    # Get SAE activations for each position
    with torch.no_grad():
        sae_features = sae.encode(layer_4_embeds)  # [seq_len, 10420]
        #print('SAE features shape:')
        #print(sae_features.shape)
    
    # Get max activation per feature across sequence
    max_activations = sae_features.max(dim=0).values  # [10420]
    
    # Get mean activation per feature
    mean_activations = sae_features.mean(dim=0)  # [10420]
    
    # Count active features (activation > threshold)
    active_features = (sae_features > threshold).any(dim=0).sum().item()
    
    return {
        'protein_id': protein_id,
        'max_activations': max_activations.cpu().numpy(),
        'mean_activations': mean_activations.cpu().numpy(),
        'n_active_features': active_features
    }

In [45]:
# Enzymes df grouped by variability
df = pd.read_csv(data_dir / "processed" / "iml1515_variability_grouped.csv")

results = []
for _, row in df.iterrows():
    if row['variance_group'] in ['high', 'low']:
        result = get_sae_activations(row['sequence'], row['gene'])
        result['variance_group'] = row['variance_group']
        result['eta_cv'] = row['eta_cv']
        results.append(result)

# Save results
with open(sae_activations_path / "sae_activations.pkl", 'wb') as f:
    pickle.dump(results, f)

## 6. Analyze results

In [46]:
# Load activation data
with open(sae_activations_path / "sae_activations.pkl", 'rb') as f:
    data = pickle.load(f)

In [47]:
# Organize into matrices
high_var = [d for d in data if d['variance_group'] == 'high']
low_var = [d for d in data if d['variance_group'] == 'low']

high_var_matrix = np.stack([d['max_activations'] for d in high_var])  # [n_high, 10420]
low_var_matrix = np.stack([d['max_activations'] for d in low_var])    # [n_low, 10420]

n_features = high_var_matrix.shape[1]
print(f"Number of features: {n_features}")
print(f"High variance proteins: {high_var_matrix.shape[0]}")
print(f"Low variance proteins: {low_var_matrix.shape[0]}")

Number of features: 10240
High variance proteins: 61
Low variance proteins: 54


In [48]:
# For each feature, test if activation differs between groups
feature_stats = []

for feature_idx in range(n_features):  
    high_activations = high_var_matrix[:, feature_idx]
    low_activations = low_var_matrix[:, feature_idx]
    
    # Mann-Whitney U test (non-parametric)
    statistic, p_value = stats.mannwhitneyu(
        high_activations, low_activations, alternative='two-sided')
    
    # Effect size (Cohen's d)
    mean_diff = high_activations.mean() - low_activations.mean()
    pooled_std = np.sqrt((high_activations.std()**2 + low_activations.std()**2) / 2)
    cohens_d = mean_diff / pooled_std if pooled_std > 0 else 0
    
    # Activation frequency
    freq_high = (high_activations > 0.15).mean()
    freq_low = (low_activations > 0.15).mean()
    
    feature_stats.append({
        'feature_id': feature_idx,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'mean_activation_high': high_activations.mean(),
        'mean_activation_low': low_activations.mean(),
        'freq_high': freq_high,
        'freq_low': freq_low,
        'freq_diff': freq_high - freq_low
    })

stats_df = pd.DataFrame(feature_stats)

# Multiple testing correction (Bonferroni)
stats_df['p_value_corrected'] = stats_df['p_value'] * len(stats_df)
stats_df['significant'] = stats_df['p_value_corrected'] < 0.05

# Rank by effect size
stats_df = stats_df.sort_values('cohens_d', key=abs, ascending=False)

print(f"\nSignificant features: {stats_df['significant'].sum()}")
print(f"Top 10 discriminating features:")
print(stats_df[['feature_id', 'cohens_d', 'p_value_corrected', 'freq_diff']].head(10))


Significant features: 0
Top 10 discriminating features:
      feature_id  cohens_d  p_value_corrected  freq_diff
669          669  0.746409           2.147111   0.000000
6704        6704  0.744882           2.733663   0.000000
7981        7981  0.741933           1.402798   0.000000
5501        5501  0.701801           4.477312   0.000000
7669        7669  0.696819          13.672544   0.237705
705          705  0.695003           1.429742   0.345173
1391        1391  0.681320           1.757405   0.039162
3764        3764  0.673699          13.941392   0.113236
1119        1119  0.671716           7.831959   0.000000
8573        8573  0.665353          16.274266   0.000000
